# SchoolBridge — LayoutXLM 자체화 PoC (Colab, 한국어)

**목표**: LLM 1단계 (Claude sentence 추출) → LayoutXLM 자체 모델로 대체. 학년 매트릭스 같은 표 의미 복원.

**v2 개선점** (이전 노트북 대비):
- `microsoft/layoutxlm-base` 사용 — 한국어 포함 multilingual (LayoutLMv2 기반)
- 한국어 토큰 정상 표시 (이전 LayoutLMv3-base는 영어라 byte 단위 깨짐)
- 한글 폰트 적용 시각화
- HWP → PDF 자동 변환 (LibreOffice)
- 학년 매트릭스 검출 진단 셀 추가

**Colab 세팅**: 런타임 → 런타임 유형 변경 → T4 GPU

## 1. 환경 체크

In [ ]:
import sys, platform
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except ImportError:
    print("PyTorch 미설치 — 다음 셀에서 설치됨")

## 2. 라이브러리 + 시스템 패키지 설치 (~2~3분)

- `transformers`, `sentencepiece` — LayoutXLM 토크나이저 (XLM-R)
- `pdfplumber`, `pymupdf` — PDF 추출/렌더링
- `fonts-nanum` — 한글 시각화
- `libreoffice` — HWP → PDF 자동 변환 (선택)

In [ ]:
!pip install -q transformers sentencepiece pdfplumber pymupdf pillow
!apt-get install -y -q fonts-nanum libreoffice > /dev/null 2>&1
!fc-cache -fv > /dev/null 2>&1
print("설치 완료")

## 3. 가정통신문 파일 업로드

**추천 파일** (백엔드 `data/` 폴더에서):
- ⭐ `2026학년도 학습준비물 안내 가정통신문.hwp` — **학년 매트릭스 케이스 (핵심)**
- `해조류박람회 체험학습 참가 동의서.hwpx` — HWPX 다중 셀
- `2026 2,3,5,6학년 구강검진 실시안내.pdf` — PDF 표 인식

한 번에 여러 개 올려도 됩니다.

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
uploaded_paths = [Path(name) for name in uploaded.keys()]
for p in uploaded_paths:
    print(f"  {p.name}: {p.stat().st_size // 1024} KB ({p.suffix})")

## 4. HWP → PDF 자동 변환 (LibreOffice headless)

한컴 HWP 5.0 파일은 LibreOffice로 PDF 변환 시도. 표 구조가 일부 손실될 수 있지만 90% 케이스 OK.

In [ ]:
import subprocess

converted = []
for p in uploaded_paths:
    if p.suffix.lower() == ".hwp":
        print(f"변환: {p.name} → PDF")
        result = subprocess.run(
            ["libreoffice", "--headless", "--convert-to", "pdf", str(p)],
            capture_output=True, text=True, timeout=60,
        )
        if result.returncode == 0:
            pdf_path = p.with_suffix(".pdf")
            if pdf_path.exists():
                print(f"  → {pdf_path.name} ({pdf_path.stat().st_size // 1024} KB)")
                converted.append(pdf_path)
            else:
                print(f"  ⚠️ 변환 실패: {result.stderr[:200]}")
        else:
            print(f"  ⚠️ {result.stderr[:200]}")

uploaded_paths.extend(converted)
print(f"\n총 처리 가능 파일: {len(uploaded_paths)}")

## 5. 텍스트 + bbox 추출

글자 위치만. 표 구조 의미는 LayoutXLM이 학습으로 처리.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

@dataclass
class TextSpan:
    text: str
    bbox: Tuple[float, float, float, float]  # normalized 0~1
    page: int = 0
    cell_id: Optional[str] = None
    is_in_table: bool = False

### 5-1. HWPX 추출

In [ ]:
import zipfile
import xml.etree.ElementTree as ET

HP_NS = "http://www.hancom.co.kr/hwpml/2011/paragraph"

def extract_hwpx(path: Path) -> List[TextSpan]:
    spans: List[TextSpan] = []
    with zipfile.ZipFile(path) as z:
        section_files = sorted(n for n in z.namelist()
                                if n.startswith("Contents/section") and n.endswith(".xml"))
        for sf in section_files:
            root = ET.fromstring(z.read(sf))
            for tbl_idx, tbl in enumerate(root.iter(f"{{{HP_NS}}}tbl")):
                for tr_idx, tr in enumerate(tbl.iter(f"{{{HP_NS}}}tr")):
                    for tc_idx, tc in enumerate(tr.iter(f"{{{HP_NS}}}tc")):
                        texts = [t.text for t in tc.iter() if t.text]
                        joined = " ".join(s.strip() for s in texts if s.strip())
                        if not joined:
                            continue
                        cellAddr = tc.find(f"{{{HP_NS}}}cellAddr")
                        row = int(cellAddr.get("rowAddr")) if cellAddr is not None and cellAddr.get("rowAddr") else tr_idx
                        col = int(cellAddr.get("colAddr")) if cellAddr is not None and cellAddr.get("colAddr") else tc_idx
                        cell_id = f"tbl{tbl_idx}_r{row}_c{col}"
                        bbox = (col * 0.12, row * 0.04, (col + 1) * 0.12, (row + 1) * 0.04)
                        spans.append(TextSpan(text=joined, bbox=bbox, cell_id=cell_id, is_in_table=True))
            seen = {s.text for s in spans}
            for p_idx, para in enumerate(root.iter(f"{{{HP_NS}}}p")):
                texts = [t.text for t in para.iter() if t.text]
                joined = " ".join(s.strip() for s in texts if s.strip())
                if joined and joined not in seen:
                    bbox = (0.0, min(0.99, p_idx * 0.02), 1.0, min(1.0, (p_idx + 1) * 0.02))
                    spans.append(TextSpan(text=joined, bbox=bbox, is_in_table=False))
                    seen.add(joined)
    return spans

### 5-2. PDF 추출 (text layer + 표 인식)

**표 인식 강화** — `find_tables` 옵션 조정으로 c0만 잡히던 이전 문제 보완

In [ ]:
import pdfplumber

TABLE_SETTINGS_STRICT = {
    "vertical_strategy": "lines",
    "horizontal_strategy": "lines",
    "snap_tolerance": 3,
    "join_tolerance": 3,
}
TABLE_SETTINGS_LOOSE = {
    "vertical_strategy": "text",
    "horizontal_strategy": "text",
    "min_words_vertical": 2,
    "min_words_horizontal": 2,
}

def extract_pdf(path: Path) -> List[TextSpan]:
    spans: List[TextSpan] = []
    with pdfplumber.open(path) as pdf:
        for page_idx, page in enumerate(pdf.pages):
            page_w, page_h = page.width, page.height
            # 표 자동 인식 — strict 먼저, 비었으면 loose
            try:
                tables = page.find_tables(table_settings=TABLE_SETTINGS_STRICT)
                if not tables:
                    tables = page.find_tables(table_settings=TABLE_SETTINGS_LOOSE)
            except Exception:
                tables = []
            table_bboxes = []
            for t_idx, table in enumerate(tables):
                for r_idx, row in enumerate(table.rows):
                    for c_idx, cell_bbox in enumerate(row.cells):
                        if cell_bbox:
                            table_bboxes.append((cell_bbox, t_idx, r_idx, c_idx))
            words = page.extract_words()
            for w in words:
                bbox = (w["x0"] / page_w, w["top"] / page_h,
                        w["x1"] / page_w, w["bottom"] / page_h)
                cell_id = None
                in_table = False
                for cb, t, r, c in table_bboxes:
                    cx0, cy0, cx1, cy1 = cb
                    # 단어 중심점이 셀 안에 있는지 확인 (경계 넘침 방지)
                    cx, cy = (w["x0"] + w["x1"]) / 2, (w["top"] + w["bottom"]) / 2
                    if cx0 <= cx <= cx1 and cy0 <= cy <= cy1:
                        cell_id = f"tbl{t}_r{r}_c{c}"
                        in_table = True
                        break
                spans.append(TextSpan(text=w["text"], bbox=bbox, page=page_idx,
                                      cell_id=cell_id, is_in_table=in_table))
    return spans

### 5-3. 파일별 추출 실행

In [ ]:
all_spans = {}
for p in uploaded_paths:
    ext = p.suffix.lower()
    if ext == ".hwpx":
        spans = extract_hwpx(p)
    elif ext == ".pdf":
        spans = extract_pdf(p)
    else:
        continue
    all_spans[p.name] = spans
    table_spans = [s for s in spans if s.is_in_table]
    text_spans = [s for s in spans if not s.is_in_table]
    unique_cells = len(set(s.cell_id for s in table_spans if s.cell_id))
    print(f"\n📄 {p.name}")
    print(f"  spans: {len(spans)} (표 {len(table_spans)} + 표 밖 {len(text_spans)})")
    print(f"  unique cells: {unique_cells}")
    if unique_cells <= 3:
        print(f"  ⚠️ 셀이 거의 안 잡힘 — 1열 단순 표거나 학년 매트릭스 아님")
    elif unique_cells > 10:
        print(f"  ✅ 다중 셀 표 — 학년 매트릭스 후보 가능성")

In [ ]:
# 첫 파일의 첫 30 span 출력
for name, spans in all_spans.items():
    print(f"=== {name} ===")
    for s in spans[:30]:
        marker = "[TBL]" if s.is_in_table else "[TXT]"
        print(f"  {marker} p{s.page} {s.cell_id or '':18s} {s.text[:60]!r}")
    print()
    break  # 첫 파일만

## 6. 페이지 이미지 렌더링

In [ ]:
import fitz
from PIL import Image

def render_pdf_pages(path: Path, dpi: int = 150) -> List[Image.Image]:
    images = []
    doc = fitz.open(path)
    for page in doc:
        pix = page.get_pixmap(dpi=dpi)
        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
        images.append(img)
    doc.close()
    return images

# 첫 PDF 파일 렌더링
pdf_paths = [p for p in uploaded_paths if p.suffix.lower() == ".pdf"]
if pdf_paths:
    sample_pdf = pdf_paths[0]
    pdf_pages = render_pdf_pages(sample_pdf, dpi=150)
    print(f"{sample_pdf.name} — {len(pdf_pages)} page(s), size {pdf_pages[0].size}")
    pdf_pages[0]

## 7. 시각화 (한글 폰트 적용)

**빨강** = 표 셀 / **파랑** = 표 밖 텍스트. bbox가 글자 위에 잘 얹혀야 추출 정확.

In [ ]:
from PIL import ImageDraw, ImageFont
import os

# 한글 폰트 경로
FONT_CANDIDATES = [
    "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
    "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
]
FONT_PATH = next((p for p in FONT_CANDIDATES if os.path.exists(p)), None)
print(f"폰트: {FONT_PATH or '없음 — 텍스트 라벨 X'}")

def overlay_bboxes(img: Image.Image, spans: List[TextSpan], page: int = 0,
                   show_text: bool = False) -> Image.Image:
    out = img.copy()
    draw = ImageDraw.Draw(out)
    W, H = out.size
    font = ImageFont.truetype(FONT_PATH, 11) if (FONT_PATH and show_text) else None
    for s in spans:
        if s.page != page:
            continue
        x0, y0, x1, y1 = s.bbox
        rect = (x0 * W, y0 * H, x1 * W, y1 * H)
        color = (220, 50, 50) if s.is_in_table else (50, 100, 220)
        draw.rectangle(rect, outline=color, width=2)
        if font and show_text:
            draw.text((rect[0], rect[1] - 14), s.text[:20], fill=color, font=font)
    return out

if pdf_paths and 'pdf_pages' in dir():
    spans = all_spans[sample_pdf.name]
    overlay = overlay_bboxes(pdf_pages[0], spans, page=0, show_text=False)
    print("빨강=표 셀, 파랑=표 밖. 박스가 글자랑 어긋나면 추출 부정확.")
    overlay

## 8. LayoutXLM 모델 로드 (한국어 multilingual)

`microsoft/layoutxlm-base` — LayoutLMv2 architecture + XLM-RoBERTa 토크나이저 (50개 언어). 한국어 단어를 1~2 토큰으로 처리.

In [ ]:
from transformers import LayoutXLMProcessor, LayoutLMv2ForTokenClassification
import torch

MODEL_ID = "microsoft/layoutxlm-base"
processor = LayoutXLMProcessor.from_pretrained(MODEL_ID, apply_ocr=False)
print("Processor:", processor.__class__.__name__)
print("Tokenizer:", processor.tokenizer.__class__.__name__)

In [ ]:
# 한국어 토큰 분할 테스트 — "학년", "준비물" 같은 단어가 잘 잡히나
test_words = ["학년", "준비물", "제출", "체험학습", "스케치북", "색연필"]
for w in test_words:
    tokens = processor.tokenizer.tokenize(w)
    print(f"  {w:8s} → {tokens}")

## 9. LayoutXLM 입력 변환 + 모델 추론

In [ ]:
LABELS = [
    "O",
    "B-HEADER_ROW", "I-HEADER_ROW",
    "B-HEADER_COL", "I-HEADER_COL",
    "B-VALUE", "I-VALUE",
    "B-FREE_TEXT", "I-FREE_TEXT",
]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

model = LayoutLMv2ForTokenClassification.from_pretrained(
    MODEL_ID,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()
print(f"Model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")
print(f"Device: {device}")

In [ ]:
def to_layoutxlm_inputs(image: Image.Image, spans: List[TextSpan], page: int = 0):
    page_spans = [s for s in spans if s.page == page]
    words = [s.text for s in page_spans]
    boxes = []
    for s in page_spans:
        x0, y0, x1, y1 = s.bbox
        boxes.append([
            max(0, min(1000, int(x0 * 1000))),
            max(0, min(1000, int(y0 * 1000))),
            max(0, min(1000, int(x1 * 1000))),
            max(0, min(1000, int(y1 * 1000))),
        ])
    encoded = processor(
        image, words, boxes=boxes,
        return_tensors="pt",
        truncation=True, padding="max_length", max_length=512,
    )
    return encoded, page_spans

if pdf_paths and 'pdf_pages' in dir():
    encoded, page_spans = to_layoutxlm_inputs(pdf_pages[0], all_spans[sample_pdf.name], page=0)
    print("Input shapes:")
    for k, v in encoded.items():
        if hasattr(v, "shape"):
            print(f"  {k}: {tuple(v.shape)}")
    print(f"페이지 단어 수: {len(page_spans)}")

In [ ]:
# 추론 (random init head — 결과는 의미 X, 파이프라인 + 한국어 토큰 검증용)
if pdf_paths and 'encoded' in dir():
    inputs = {k: v.to(device) for k, v in encoded.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    predictions = outputs.logits.argmax(-1)[0].tolist()
    tokens = processor.tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    print("한국어 토큰 + random init 라벨 (파이프라인 검증):")
    cnt = 0
    for tok, pred in zip(tokens, predictions):
        if tok in ("<s>", "</s>", "<pad>"):
            continue
        clean_tok = tok.replace("▁", " ").strip() or tok
        print(f"  {clean_tok:25s} → {ID2LABEL[pred]}")
        cnt += 1
        if cnt >= 30:
            break
    print("\n→ 한국어 토큰이 정상 표시되면 LayoutXLM 셋업 OK")
    print("→ 라벨은 random init이라 의미 없음. fine-tune 후 실제 헤더/값 구분 됨.")

## 10. 학년 매트릭스 진단

업로드한 파일이 학년 매트릭스를 포함하는지 검사. "학년", "1", "2", ..., "6" 같은 패턴이 같은 행에 나란히 있으면 매트릭스 케이스.

In [ ]:
from collections import Counter

def diagnose_matrix(spans: List[TextSpan]) -> dict:
    """학년 매트릭스 패턴 검출"""
    table_spans = [s for s in spans if s.is_in_table and s.cell_id]
    # 같은 row의 셀들 그룹
    rows = {}
    for s in table_spans:
        try:
            r = int(s.cell_id.split("_r")[1].split("_")[0])
            c = int(s.cell_id.split("_c")[1])
        except Exception:
            continue
        rows.setdefault(r, []).append((c, s.text))
    # 학년 패턴 찾기: 한 행에 1~6 또는 1학년~6학년이 순서대로
    grade_patterns = ["1", "2", "3", "4", "5", "6",
                       "1학년", "2학년", "3학년", "4학년", "5학년", "6학년"]
    matrix_rows = []
    for r, cells in rows.items():
        cells_sorted = sorted(cells)
        texts = [t for _, t in cells_sorted]
        grade_hits = [t for t in texts if t.strip() in grade_patterns]
        if len(grade_hits) >= 3:
            matrix_rows.append((r, texts, grade_hits))
    return {
        "total_rows": len(rows),
        "total_cells": len(table_spans),
        "max_col": max((c for cells in rows.values() for c, _ in cells), default=0) + 1,
        "matrix_candidates": matrix_rows,
    }

for name, spans in all_spans.items():
    diag = diagnose_matrix(spans)
    print(f"\n📊 {name}")
    print(f"  표 행 수: {diag['total_rows']}, 셀 수: {diag['total_cells']}, 최대 열: {diag['max_col']}")
    if diag["matrix_candidates"]:
        print(f"  ⭐ 학년 매트릭스 후보 {len(diag['matrix_candidates'])}개 발견:")
        for r, texts, hits in diag["matrix_candidates"][:3]:
            print(f"    row {r}: {' | '.join(texts[:10])}")
    else:
        print(f"  단순 표 (학년 매트릭스 X)")

## 11. 다음 단계

**여기까지 통과한 것**:
1. ✅ HWPX/PDF 추출 (LibreOffice로 HWP→PDF도)
2. ✅ pdfplumber 표 인식 (strict + loose fallback)
3. ✅ LayoutXLM 한국어 토큰 분할 정상 ("학년" → 1~2토큰)
4. ✅ 모델 입력 형식 + GPU 추론
5. ✅ 학년 매트릭스 진단 셀

**남은 단계** (실제 자체화):
1. ⏳ 라벨링 도구 셋업 (Label Studio) — 1,000장 라벨링 30~50시간
2. ⏳ Fine-tune (Colab Pro / NCP GPU L4, 5 epoch ~6시간)
3. ⏳ 평가 — Claude 1단계 vs LayoutXLM 동일 통신문 비교
4. ⏳ NCP CPU inference 통합

**핵심**: 우리는 자체 모델 3개(KoELECTRA·KcELECTRA·NLLB) 이미 보유 → LayoutXLM 1개 추가로 **API 의존 0** 완전 자체 파이프라인. 스쿨포인트처럼 전 단계 API 의존인 서비스는 이 경로 자체가 막힘.